## celda 1 - cargar con Usuario_Id

In [1]:
import pandas as pd
from pathlib import Path

CARPETA = Path("datos")

def convertir_fecha(serie):
    fecha = pd.to_datetime(serie, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    corto = fecha.isna()
    fecha.loc[corto] = pd.to_datetime(serie[corto], format="%d/%m/%Y %H:%M", errors="coerce")
    return fecha

partes = []
for ruta in sorted(CARPETA.glob("20*/datos_abiertos_*.csv")):
    df = pd.read_csv(ruta, encoding="latin-1", usecols=[1, 4, 5, 6, 7], dtype=str)
    df.columns = ["usuario", "inicio", "fin", "origen", "destino"]
    partes.append(df)
u = pd.concat(partes, ignore_index=True)
del partes

u["inicio"] = convertir_fecha(u["inicio"])
u["fin"] = convertir_fecha(u["fin"])
u["origen"] = u["origen"].astype(int)
u["destino"] = u["destino"].astype(int)
u["dur"] = (u["fin"] - u["inicio"]).dt.total_seconds() / 60
u = u[(u["dur"] >= 1) & (u["dur"] <= 1440)].copy()   # misma limpieza del paso 4
print(f"Viajes: {len(u):,} | Usuarios distintos: {u['usuario'].nunique():,}")

Viajes: 8,916,829 | Usuarios distintos: 42,892


## celda 2 - usuarios con un numero anormal de viajes

In [2]:
u["dia"] = u["inicio"].dt.normalize()
por_dia = u.groupby(["usuario", "dia"]).size()

print("Viajes por usuario en un mismo día:")
print(por_dia.describe(percentiles=[.5, .9, .99, .999]).round(2))
for umbral in (10, 20, 50):
    print(f"Usuario-día con más de {umbral} viajes: {(por_dia > umbral).sum():,}")

print("\nLos 10 usuario-día con más viajes:")
print(por_dia.nlargest(10).to_string())

top = u["usuario"].value_counts().head(10)
print("\nLos 10 usuarios con más viajes en 2024-2025:")
print(top.to_string())
print(f"Concentran el {top.sum() / len(u):.2%} de todos los viajes")

Viajes por usuario en un mismo día:
count    4265453.00
mean           2.09
std            1.42
min            1.00
50%            2.00
90%            4.00
99%            7.00
99.9%         12.00
max           55.00
dtype: float64
Usuario-día con más de 10 viajes: 8,305
Usuario-día con más de 20 viajes: 518
Usuario-día con más de 50 viajes: 3

Los 10 usuario-día con más viajes:
usuario  dia       
764024   2024-12-16    55
         2024-12-08    51
         2025-01-10    51
641666   2024-12-20    50
764024   2024-12-15    48
641666   2024-12-19    46
3375342  2024-03-10    43
         2024-04-11    43
764024   2024-11-03    43
3375342  2024-04-12    42

Los 10 usuarios con más viajes en 2024-2025:
usuario
501066     7288
3353254    6942
1723532    6026
1140415    5386
2274063    5333
434112     5261
47146      5174
534512     4923
546406     4915
1220241    4465
Concentran el 0.62% de todos los viajes


## celda 3 - ¿Quien genero las horas extremas?

In [3]:
tabla = pd.read_parquet("procesado/tabla_modelo.parquet")
extremos = tabla.nlargest(10, "salidas")[["origen", "hora_inicio", "salidas"]]
u["hora_inicio"] = u["inicio"].dt.floor("h")

filas = []
for _, r in extremos.iterrows():
    g = u[(u["origen"] == r["origen"]) & (u["hora_inicio"] == r["hora_inicio"])]
    filas.append({
        "origen": r["origen"],
        "hora": r["hora_inicio"],
        "salidas": len(g),
        "usuarios_distintos": g["usuario"].nunique(),
        "viajes_del_usuario_top": g["usuario"].value_counts().iloc[0],
        "destinos_distintos": g["destino"].nunique(),
        "duracion_mediana_min": round(g["dur"].median(), 1),
    })
print(pd.DataFrame(filas).to_string(index=False))

 origen                hora  salidas  usuarios_distintos  viajes_del_usuario_top  destinos_distintos  duracion_mediana_min
    194 2025-09-20 21:00:00      205                 197                       2                  45                  22.8
    194 2025-09-20 20:00:00      173                 167                       2                  30                  24.5
    194 2025-11-01 21:00:00      106                 104                       2                  29                  24.0
     50 2025-09-20 20:00:00      103                 100                       2                  27                  23.8
    196 2025-09-20 20:00:00       90                  86                       2                  24                  19.3
     50 2025-09-20 21:00:00       88                  84                       2                  37                  19.7
     50 2025-11-01 20:00:00       83                  72                       3                  25                  18.4
     51 2024-02-